# Real JEV · Quickstart

One authenticated request to TypeSafe's hosted JEV model. No local LLM or GPU.
The call uses your provider quota. Store `TYPESAFE_API_KEY` in Colab Secrets; never paste it into a saved cell.

This notebook is an independent integration, not a reproduction of TypeSafe model weights or training.


In [ ]:
%pip -q install typesafe-sdk


In [ ]:
import os, getpass, time
from typesafe_sdk import TypeSafeClient, Choice, Score, Noul

key = os.environ.get("TYPESAFE_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("TYPESAFE_API_KEY")
    except Exception:
        key = None
if not key:
    key = getpass.getpass("TypeSafe API key: ").strip()
if not key:
    raise ValueError("A TypeSafe API key is required. No local fallback is used.")
os.environ["TYPESAFE_API_KEY"] = key
del key
client = TypeSafeClient()


In [ ]:
state = "Our account was billed twice. Please refund the duplicate payment."
started = time.perf_counter()
response = client.system_one(state=state, questions={
    "team": Choice(instructions="Which team should handle this request?", criteria={
        "billing": "Invoices and payments", "technical": "Product faults"}),
    "impact": Score(instructions="Rate the reported operational impact.", criteria=[
        "No disruption", "Work slowed", "Work blocked"]),
    "refund": Noul(instructions="Does the sender explicitly request a refund?"),
})
print("Backend: TypeSafe JEV (hosted)")
print(f"Wall-clock request latency: {(time.perf_counter()-started)*1000:.1f} ms")
print(response)
team = response.answers["team"]
threshold = 0.8  # Example only: evaluate thresholds on your own data.
route = team.choice if team.confidence >= threshold else "human_review"
print("Application route:", route)


## Interpret the result

Confidence and option probability are separate provider outputs. The threshold above is illustrative.
A single request does not establish latency percentiles, calibration or decision accuracy.
API errors propagate rather than producing mock answers or switching to a different model.
Clear outputs before sharing a notebook containing private state.

[Official API documentation](https://docs.typesafe.ai/introduction/quickstart) · [Repository](https://github.com/vtavakkoli/simple-jev)
